# 📊 Export Data & Perhitungan Detail (TF-IDF & SAW)

Notebook ini khusus dibuat untuk mengekspor hasil dari proses **TF-IDF** dan perhitungan **SAW (Simple Additive Weighting)** secara bertahap ke dalam format `.csv`.

### Bobot Kriteria
| Kode | Kriteria | Jenis | Bobot |
|------|----------|-------|-------|
| C1 | Fasilitas Hotel | Benefit | 0.45 |
| C2 | Score Sentiment | Benefit | 0.30 |
| C3 | Rating | Benefit | 0.25 |

In [21]:
import pandas as pd
import os
from sklearn.feature_extraction.text import TfidfVectorizer

# Buat folder output jika belum ada
os.makedirs('output_csv', exist_ok=True)

## 1. Hasil TF-IDF (Term Frequency - Inverse Document Frequency)
Mengekstrak teks ulasan menjadi matriks TF-IDF. Nilai TF-IDF sudah ternormalisasi dalam skala 0–1.

In [22]:
data_ulasan = {
    'text': [
        'Pengalaman menginap yang sangat luar biasa. Kamarnya luas, bersih, dan wanginya sangat menenangkan setelah seharian beraktivitas di luar. Pasti akan kembali lagi!',
        'Sangat kecewa dengan pelayanannya. Saat check-in resepsionis terlihat judes, dan air panas di kamar mandi sama sekali tidak menyala meskipun sudah komplain dua kali.',
        'Fasilitas hotel cukup standar untuk harga yang ditawarkan. Kolam renangnya lumayan bersih, tapi menu sarapannya kurang bervariasi dan rasanya biasa saja.',
        'The location is absolutely perfect, right in the heart of the city with easy access to public transport. The breakfast buffet was exceptional with lots of healthy options.',
        'Terrible experience from start to finish. The room smelled like smoke even though I specifically requested a non-smoking room, and the bed sheets looked stained.',
        'Hotel yang sangat cocok untuk liburan keluarga. Anak-anak sangat senang dengan fasilitas kids club dan kolam renang air hangatnya. Staf hotel juga sangat ramah dan sabar.',
        'Sayang sekali AC di kamar saya bocor dan menetes membasahi karpet, membuat ruangan jadi bau lembap. Proses perbaikan juga memakan waktu sangat lama.',
        'Not a bad place to stay for a quick business trip. Wi-Fi connection was stable and fast enough for video calls, though the room lighting was a bit too dim to work comfortably.',
        'Harganya lumayan mahal tapi sayang sekali kebersihan kamarnya kurang diperhatikan. Masih ada debu di meja nakas dan handuknya terasa kasar seperti sudah sangat lama.',
        'Proses check-in dan check-out sangat mulus dan cepat. Staf concierge sangat informatif dalam memberikan rekomendasi tempat wisata lokal. Pelayanan bintang lima!'
    ]
}
df_ulasan = pd.DataFrame(data_ulasan)

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df_ulasan['text'])
feature_names = vectorizer.get_feature_names_out()

df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
df_tfidf.insert(0, 'Text Asli', df_ulasan['text'])

print('==== HASIL MATRIKS TF-IDF ====')
print(f'Jumlah dokumen: {len(df_tfidf)} | Jumlah fitur (kata unik): {len(feature_names)}')
display(df_tfidf.head())

df_tfidf.to_csv('output_csv/1_tfidf_results.csv', index=False)
print('\nBerhasil diekspor ke: output_csv/1_tfidf_results.csv')

==== HASIL MATRIKS TF-IDF ====
Jumlah dokumen: 10 | Jumlah fitur (kata unik): 180


,Text Asli,absolutely,ac,access,ada,air,akan,anak,and,bad,...,untuk,video,waktu,wanginya,was,wi,wisata,with,work,yang
0,Pengalaman menginap yang sangat luar biasa. Ka...,0.000000,0.0,0.000000,0.0,0.000000,0.220601,0.0,0.000000,0.0,...,0.000000,0.0,0.0,0.220601,0.000000,0.0,0.0,0.000000,0.0,0.164068
1,Sangat kecewa dengan pelayanannya. Saat check-...,0.000000,0.0,0.000000,0.0,0.186923,0.000000,0.0,0.000000,0.0,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000
2,Fasilitas hotel cukup standar untuk harga yang...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,...,0.204414,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.178838
3,"The location is absolutely perfect, right in t...",0.167415,0.0,0.167415,0.0,0.000000,0.000000,0.0,0.000000,0.0,...,0.000000,0.0,0.0,0.000000,0.142318,0.0,0.0,0.334829,0.0,0.000000
4,Terrible experience from start to finish. The ...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.173158,0.0,...,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000



Berhasil diekspor ke: output_csv/1_tfidf_results.csv


---
## 2. Algoritma SAW — Tahap 2.1: Matriks Keputusan (Nilai Mentah)
Nilai mentah adalah nilai asli sebelum diproses lebih lanjut:
- **C1 (Fasilitas)**: dihitung dari jumlah poin kategori fasilitas (skala 0–0.45)
- **C2 (Sentimen)**: rata-rata sentimen ulasan dari model ML (skala 0–100)
- **C3 (Rating)**: rata-rata rating bintang dari tamu (skala 0–10)

> **Catatan**: Nilai mentah C2 dan C3 sengaja masih dalam skala aslinya. Penyeragaman skala dilakukan di tahap normalisasi.

In [23]:
hotels = [
    {'name': 'Hotel Indah Jaya',         'favoriteFeatures': ['Free Wi-Fi in all rooms!', 'Breakfast [free]', 'Air conditioning', 'Swimming pool'],                                                       'averageSentimentScore': 85.0, 'averageRating': 8.8},
    {'name': 'Penginapan Sederhana',      'favoriteFeatures': ['Free Wi-Fi in all rooms!', 'Air conditioning'],                                                                                             'averageSentimentScore': 60.0, 'averageRating': 6.5},
    {'name': 'Luxury Resort Spa',         'favoriteFeatures': ['Free Wi-Fi in all rooms!', 'Restaurant [halal]', 'Room service', 'Airport transfer', 'Massage', 'Elevator'],                               'averageSentimentScore': 92.5, 'averageRating': 9.4},
    {'name': 'Budget Inn City Center',    'favoriteFeatures': ['Free Wi-Fi in all rooms!', 'Check-in/out [express]'],                                                                                       'averageSentimentScore': 50.5, 'averageRating': 5.8},
    {'name': 'Grand Emerald Suites',      'favoriteFeatures': ['Free Wi-Fi in all rooms!', 'Breakfast [free]', 'Fitness center', 'Swimming pool', 'Shuttle service', 'Elevator', 'Shared lounge/TV area'], 'averageSentimentScore': 88.0, 'averageRating': 9.0},
    {'name': 'Sea View Beach Hotel',      'favoriteFeatures': ['Free Wi-Fi in all rooms!', 'Restaurants', 'Private beach', 'Room service', 'Balcony/terrace'],                                             'averageSentimentScore': 78.5, 'averageRating': 8.2},
    {'name': 'Transit Hotel Airport',     'favoriteFeatures': ['Free Wi-Fi in all rooms!', 'Airport transfer', 'Room service [24-hour]', 'Cash withdrawal'],                                               'averageSentimentScore': 65.0, 'averageRating': 7.0},
    {'name': 'Boutique Art Hotel',        'favoriteFeatures': ['Free Wi-Fi in all rooms!', 'Coffee shop', 'Garden', 'Safety deposit boxes'],                                                               'averageSentimentScore': 82.0, 'averageRating': 8.5},
    {'name': 'Eco Lodge Resort',          'favoriteFeatures': ['Restaurant [halal]', 'Hiking', 'Tours', 'Garden'],                                                                                         'averageSentimentScore': 75.0, 'averageRating': 8.0},
    {'name': 'The Presidential Palace',   'favoriteFeatures': ['Free Wi-Fi in all rooms!', 'Restaurant [halal]', 'Room service', 'Massage', 'Sauna', 'Elevator', 'Concierge', 'Smoking area'],            'averageSentimentScore': 96.0, 'averageRating': 9.8},
]

def calculate_c1(features):
    """Hitung skor C1 berdasarkan kategori fasilitas. Maksimum = 0.45"""
    score = 0.0
    features = [f.lower() for f in features]
    if any('wifi' in f or 'wi-fi' in f for f in features):                                                                                           score += 0.10  # Internet
    if any('breakfast' in f or 'restaurant' in f or 'coffee shop' in f for f in features):                                                           score += 0.15  # F&B
    if any('room service' in f or 'air conditioning' in f or 'balcony' in f for f in features):                                                      score += 0.10  # Kenyamanan
    if any('airport' in f or 'shuttle' in f for f in features):                                                                                      score += 0.05  # Transportasi
    if any('pool' in f or 'massage' in f or 'beach' in f or 'fitness' in f or 'sauna' in f or 'garden' in f or 'hiking' in f or 'tours' in f for f in features): score += 0.03  # Rekreasi
    if any('elevator' in f or 'check-in' in f or 'lounge' in f or 'cash' in f or 'safety' in f or 'concierge' in f or 'smoking' in f for f in features):        score += 0.02  # Lainnya
    return score

for h in hotels:
    h['c1_raw'] = calculate_c1(h['favoriteFeatures'])
    h['c2_raw'] = h['averageSentimentScore']   # Skala 0-100
    h['c3_raw'] = h['averageRating']           # Skala 0-10

df_raw = pd.DataFrame(hotels)[['name', 'c1_raw', 'c2_raw', 'c3_raw']]

print('==== TAHAP 2.1: MATRIKS KEPUTUSAN (NILAI MENTAH) ====')
print('Catatan: C2 skala 0-100, C3 skala 0-10 (belum dinormalisasi)')
display(df_raw)

df_raw.to_csv('output_csv/2_1_saw_raw_matrix.csv', index=False)
print('\nBerhasil diekspor ke: output_csv/2_1_saw_raw_matrix.csv')

==== TAHAP 2.1: MATRIKS KEPUTUSAN (NILAI MENTAH) ====
Catatan: C2 skala 0-100, C3 skala 0-10 (belum dinormalisasi)


,name,c1_raw,c2_raw,c3_raw
0,Hotel Indah Jaya,0.38,85.0,8.8
1,Penginapan Sederhana,0.20,60.0,6.5
2,Luxury Resort Spa,0.45,92.5,9.4
3,Budget Inn City Center,0.12,50.5,5.8
4,Grand Emerald Suites,0.35,88.0,9.0
5,Sea View Beach Hotel,0.38,78.5,8.2
6,Transit Hotel Airport,0.27,65.0,7.0
7,Boutique Art Hotel,0.30,82.0,8.5
8,Eco Lodge Resort,0.18,75.0,8.0
9,The Presidential Palace,0.40,96.0,9.8



Berhasil diekspor ke: output_csv/2_1_saw_raw_matrix.csv


## Tahap 2.2: Normalisasi Matriks
Menyamakan skala semua kriteria ke rentang **0 – 1** menggunakan rumus Benefit:

$$R_{ij} = \frac{X_{ij}}{\max_j(X_{ij})}$$

Setelah normalisasi, **semua nilai C1, C2, C3 pasti berada di antara 0 dan 1**.

In [24]:
max_c1 = df_raw['c1_raw'].max()
max_c2 = df_raw['c2_raw'].max()
max_c3 = df_raw['c3_raw'].max()

print(f'Nilai maksimum: max_C1={max_c1} | max_C2={max_c2} | max_C3={max_c3}')
print()

df_norm = df_raw.copy()
df_norm['c1_norm'] = df_norm['c1_raw'] / max_c1   # Hasil: 0 - 1
df_norm['c2_norm'] = df_norm['c2_raw'] / max_c2   # Hasil: 0 - 1
df_norm['c3_norm'] = df_norm['c3_raw'] / max_c3   # Hasil: 0 - 1

df_norm_view = df_norm[['name', 'c1_norm', 'c2_norm', 'c3_norm']]

print('==== TAHAP 2.2: MATRIKS NORMALISASI (Skala 0 - 1) ====')
display(df_norm_view)

df_norm_view.to_csv('output_csv/2_2_saw_normalized_matrix.csv', index=False)
print('\nBerhasil diekspor ke: output_csv/2_2_saw_normalized_matrix.csv')

Nilai maksimum: max_C1=0.44999999999999996 | max_C2=96.0 | max_C3=9.8

==== TAHAP 2.2: MATRIKS NORMALISASI (Skala 0 - 1) ====


,name,c1_norm,c2_norm,c3_norm
0,Hotel Indah Jaya,0.844444,0.885417,0.897959
1,Penginapan Sederhana,0.444444,0.625000,0.663265
2,Luxury Resort Spa,1.000000,0.963542,0.959184
3,Budget Inn City Center,0.266667,0.526042,0.591837
4,Grand Emerald Suites,0.777778,0.916667,0.918367
5,Sea View Beach Hotel,0.844444,0.817708,0.836735
6,Transit Hotel Airport,0.600000,0.677083,0.714286
7,Boutique Art Hotel,0.666667,0.854167,0.867347
8,Eco Lodge Resort,0.400000,0.781250,0.816327
9,The Presidential Palace,0.888889,1.000000,1.000000



Berhasil diekspor ke: output_csv/2_2_saw_normalized_matrix.csv


## Tahap 2.3: Perkalian Bobot (Nilai Preferensi per Kriteria)
Setiap nilai normalisasi dikalikan dengan bobotnya masing-masing:

$$V_{C1i} = W_{C1} \times R_{C1i}, \quad V_{C2i} = W_{C2} \times R_{C2i}, \quad V_{C3i} = W_{C3} \times R_{C3i}$$

Batas atas yang **tidak bisa dilampaui**:
- `c1_weighted` ≤ **0.45**
- `c2_weighted` ≤ **0.30**
- `c3_weighted` ≤ **0.25**

In [25]:
W_C1, W_C2, W_C3 = 0.45, 0.30, 0.25

df_weighted = df_norm.copy()
df_weighted['c1_weighted'] = W_C1 * df_weighted['c1_norm']   # Maks: 0.45
df_weighted['c2_weighted'] = W_C2 * df_weighted['c2_norm']   # Maks: 0.30
df_weighted['c3_weighted'] = W_C3 * df_weighted['c3_norm']   # Maks: 0.25

df_weighted_view = df_weighted[['name', 'c1_weighted', 'c2_weighted', 'c3_weighted']]

print('==== TAHAP 2.3: NILAI PREFERENSI PER KRITERIA (Setelah Dikali Bobot) ====')
print(f'  c1_weighted maks = {df_weighted["c1_weighted"].max():.4f} (threshold: 0.45)')
print(f'  c2_weighted maks = {df_weighted["c2_weighted"].max():.4f} (threshold: 0.30)')
print(f'  c3_weighted maks = {df_weighted["c3_weighted"].max():.4f} (threshold: 0.25)')
print()
display(df_weighted_view)

df_weighted_view.to_csv('output_csv/2_3_saw_weighted_per_criteria.csv', index=False)
print('\nBerhasil diekspor ke: output_csv/2_3_saw_weighted_per_criteria.csv')

==== TAHAP 2.3: NILAI PREFERENSI PER KRITERIA (Setelah Dikali Bobot) ====
  c1_weighted maks = 0.4500 (threshold: 0.45)
  c2_weighted maks = 0.3000 (threshold: 0.30)
  c3_weighted maks = 0.2500 (threshold: 0.25)



,name,c1_weighted,c2_weighted,c3_weighted
0,Hotel Indah Jaya,0.38,0.265625,0.224490
1,Penginapan Sederhana,0.20,0.187500,0.165816
2,Luxury Resort Spa,0.45,0.289062,0.239796
3,Budget Inn City Center,0.12,0.157812,0.147959
4,Grand Emerald Suites,0.35,0.275000,0.229592
5,Sea View Beach Hotel,0.38,0.245312,0.209184
6,Transit Hotel Airport,0.27,0.203125,0.178571
7,Boutique Art Hotel,0.30,0.256250,0.216837
8,Eco Lodge Resort,0.18,0.234375,0.204082
9,The Presidential Palace,0.40,0.300000,0.250000



Berhasil diekspor ke: output_csv/2_3_saw_weighted_per_criteria.csv


## Tahap 2.4: Perhitungan V-Score (Skor Preferensi Akhir) & Ranking
V-Score adalah jumlah dari semua nilai preferensi per kriteria:

$$V_i = c1\_weighted_i + c2\_weighted_i + c3\_weighted_i$$

**Nilai V-Score maksimal = 0.45 + 0.30 + 0.25 = 1.0**

In [26]:
df_final = df_weighted.copy()
df_final['saw_score'] = df_final['c1_weighted'] + df_final['c2_weighted'] + df_final['c3_weighted']
df_final['rank'] = df_final['saw_score'].rank(ascending=False).astype(int)

df_final_view = df_final.sort_values('rank')[['rank', 'name', 'saw_score', 'c1_weighted', 'c2_weighted', 'c3_weighted']]

print('==== TAHAP 2.4: HASIL AKHIR V-SCORE & RANKING ====')
print(f'  saw_score tertinggi = {df_final["saw_score"].max():.6f} (maks teoritis: 1.0)')
print()
display(df_final_view)

df_final_view.to_csv('output_csv/2_4_saw_final_ranking.csv', index=False)
print('\nBerhasil diekspor ke: output_csv/2_4_saw_final_ranking.csv')

==== TAHAP 2.4: HASIL AKHIR V-SCORE & RANKING ====
  saw_score tertinggi = 0.978858 (maks teoritis: 1.0)



,rank,name,saw_score,c1_weighted,c2_weighted,c3_weighted
2,1,Luxury Resort Spa,0.978858,0.45,0.289062,0.239796
9,2,The Presidential Palace,0.950000,0.40,0.300000,0.250000
0,3,Hotel Indah Jaya,0.870115,0.38,0.265625,0.224490
4,4,Grand Emerald Suites,0.854592,0.35,0.275000,0.229592
5,5,Sea View Beach Hotel,0.834496,0.38,0.245312,0.209184
7,6,Boutique Art Hotel,0.773087,0.30,0.256250,0.216837
6,7,Transit Hotel Airport,0.651696,0.27,0.203125,0.178571
8,8,Eco Lodge Resort,0.618457,0.18,0.234375,0.204082
1,9,Penginapan Sederhana,0.553316,0.20,0.187500,0.165816
3,10,Budget Inn City Center,0.425772,0.12,0.157812,0.147959



Berhasil diekspor ke: output_csv/2_4_saw_final_ranking.csv
